In [4]:
import sys
import uuid

# uuid_utils ko override karke standard Python uuid library use karwana
sys.modules['uuid_utils'] = uuid
sys.modules['uuid_utils.compat'] = uuid

from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import MemorySaver

In [7]:
from langgraph.graph.message import add_messages

class ChatState(TypedDict):

    messages: Annotated[list[BaseMessage], add_messages]

In [ ]:
# Not working because of API Key
llm = ChatOpenAI()

def chat_node(state: ChatState):

    # take user query from state
    messages = state['messages']

    # sent to llm
    response = ChatOpenAI.invoke(messages)

    # response store in state
    return {'messages': [response]}

In [ ]:
checkpointer = MemorySaver()

# It's also not working becasue of above cell API Key
graph = StateGraph(ChatState)

# add node
graph.add_node('chat_node', chat_node)
graph.add_edge(START, 'chat_node')
graph.add_edge('chat_node', END)

chatbot = graph.compile(checkpointer=checkpointer)

In [ ]:
chatbot

In [ ]:
initial_state = {
    'messages': [HumanMessage(content='What is the capital of Pakistan')]
}

chatbot.invoke(initial_state)['messages'][-1].content

In [ ]:
thread_id = '1'


while True:

    user_message = input("Type here: ")

    print('User: ', user_message)

    if user_message.strip().lower() in ['exit', 'quit', 'bye', 'end']:
        break

    config = {'configurable': {'thread_id': thread_id}}

    response = chatbot.invoke({'message': [HumanMessage(content=user_message)]}, config=config)

    print('AI:', response['messages'][-1].content)